## GlobalGates 추천 커넥션 시스템

유저 자기소개글(bio), 관심 카테고리, 기존 팔로우 관계를 바탕으로  
GlobalGates 안에서 **거래로 이어질 가능성이 높은 연결 후보**를 추천하는 1차 로직을 검증한다.

이 추천은
- 메인 화면 사이드바의 `팔로우 추천`
- `/friends`의 `추천 커넥션`

두 위치에 공통으로 적용할 수 있는 형태를 목표로 한다.


In [ ]:
# 최초 1회만 필요하면 실행
# %pip install pandas sqlalchemy psycopg2-binary python-dotenv scikit-learn konlpy joblib


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. .env 파일 로드
load_dotenv()

# 2. 환경 변수에서 DB 정보 가져오기
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# 3. PostgreSQL 연결 URL 생성
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 4. 엔진 생성 및 회원 데이터 불러오기
engine = create_engine(db_url)

df = pd.read_sql("SELECT id, member_nickname, member_bio FROM tbl_member", engine)

df


### 왜 추천 커넥션이 필요한가

GlobalGates는 단순 커뮤니티가 아니라, 연결이 거래로 이어지는 비즈니스 소셜 플랫폼이다.  
하지만 처음 들어온 사용자는 누구를 팔로우해야 할지 판단하기 어렵다.

현재 추천은 최신 가입순에 가까워서
- 관심 분야가 반영되지 않고
- 무역 역할 차이가 구분되지 않으며
- 실제로 연결될 만한 사람을 찾는 데 한계가 있다.

그래서 `bio + category` 유사도를 본 추천으로 두고, 완전 cold start만 별도 fallback으로 처리한다.


### 1. 추천에 사용할 실서비스 데이터를 불러온다

먼저 회원 기본 프로필을 불러온 뒤, 추천에 필요한 보조 테이블을 추가로 읽는다.

- `tbl_member`: 회원 기본 프로필과 bio
- `tbl_follow`: 기존 팔로우 관계
- `tbl_member_category_rel`: 회원 관심 카테고리

즉, 외부 데이터 없이도 플랫폼 내부 정보만으로 추천 커넥션을 만들 수 있는지 먼저 본다.


In [ ]:
follow_df = pd.read_sql("SELECT follower_id, following_id FROM tbl_follow", engine)
category_map_df = pd.read_sql(
    """
    SELECT mcr.member_id, c.category_name
    FROM tbl_member_category_rel mcr
    JOIN tbl_category c ON c.id = mcr.category_id
    """,
    engine,
)

print("member:", df.shape)
print("follow:", follow_df.shape)
print("category:", category_map_df.shape)

display(category_map_df)
follow_df.head()


### 2. 회원 데이터 상태를 먼저 점검한다

추천 로직은 텍스트 기반이라 `bio` 결측치와 중복 회원 여부를 먼저 확인해야 한다.


In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

### 3. 팔로워 수 기반 보조 지표를 만든다

이번 노트북의 메인 추천은 `bio + category` 유사도 기반으로 만든다.  
팔로워 수는 추천 본 점수에는 직접 반영하지 않고,  
bio와 카테고리 정보가 전혀 없는 완전 cold start 유저에게만 fallback 용도로 사용한다.


In [ ]:
import numpy as np

popularity = follow_df.groupby("following_id").size()

# 로그 스케일로 인기 편중 완화 후 0~5 정규화
log_pop = np.log1p(popularity)

if len(log_pop) > 0 and log_pop.max() != log_pop.min():
    score_norm = (log_pop - log_pop.min()) / (log_pop.max() - log_pop.min()) * 5
else:
    score_norm = log_pop * 0

df = df.merge(score_norm.rename("score"), left_on="id", right_index=True, how="left")
df["score"] = df["score"].fillna(0)

df


### 4. 관심 카테고리를 회원 텍스트에 합친다

bio만으로는 무역 분야가 충분히 드러나지 않는 유저가 있다.  
그래서 회원이 선택한 카테고리 이름도 추천용 텍스트에 포함한다.


In [ ]:
from konlpy.tag import Okt

okt = Okt()

category_text_df = (
    category_map_df.groupby("member_id")["category_name"]
    .apply(lambda x: " ".join(sorted(set(x))))
    .reset_index(name="category_text")
)

df = df.merge(category_text_df, left_on="id", right_on="member_id", how="left")
df = df.drop(columns=["member_id"], errors="ignore")
df["member_bio"] = df["member_bio"].fillna("")
df["category_text"] = df["category_text"].fillna("")

df["bio_text"] = df["member_bio"].apply(lambda x: " ".join(okt.nouns(x)) if x else "")
df["intro_text"] = (df["bio_text"] + " " + df["category_text"]).str.strip()

pre_df = df[["intro_text"]].rename(columns={"intro_text": "intro"})

pre_df


### 5. 인기도 점수 분포를 확인한다

팔로워 수 원본과 정규화한 `score` 분포를 비교해서  
인기 편중이 어느 정도 완화됐는지 본다.


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(popularity.values, bins=20)
axes[0].set_title("팔로워 수 원본")
axes[1].hist(df["score"].values, bins=20)
axes[1].set_title("인기도 (로그 + 0~5 정규화)")
plt.tight_layout()
plt.show()


### 6. TF-IDF + Okt 명사 추출로 회원 간 유사도를 구한다

한글 bio는 조사 영향이 크기 때문에 Okt 명사 추출 후 TF-IDF를 적용한다.  
이렇게 만든 벡터를 바탕으로 회원 간 코사인 유사도를 계산한다.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from konlpy.tag import Okt

okt = Okt()

tfidf_v = TfidfVectorizer()
tfidf_matrix = tfidf_v.fit_transform(pre_df["intro"])
cosine_full = cosine_similarity(tfidf_matrix)

print(cosine_full.shape)


### 7. 메인 추천 — 기존 유저 기준

이미 활동 중인 회원 1명을 기준으로,
- 본인 제외
- 이미 팔로우한 사람 제외

조건을 적용한 뒤 메인 화면에 보여줄 Top-5 추천 결과를 확인한다.


In [ ]:
member_id = int(df["id"].iloc[1])
self_idx = np.where(df["id"].values == member_id)[0][0]

sim_scores = cosine_full[self_idx]
sim_indices = sim_scores.argsort()[::-1]

# 본인 + 이미 팔로우한 사람 제외
followed_ids = follow_df[follow_df.follower_id == member_id].following_id.values
exclude_ids = np.append(followed_ids, member_id)

mask = ~np.isin(df.iloc[sim_indices]["id"].values, exclude_ids)
top5_indices = sim_indices[mask][:5]

print(f"기준 유저 ID: {member_id}")
print(f"추천 유저 ID: {df.iloc[top5_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[top5_indices]}")

df.iloc[top5_indices][["id", "member_nickname", "member_bio", "category_text"]]


### 8. 신규 가입 후 bio 작성

아직 팔로우 이력은 없지만 자기소개를 작성한 신규 유저라고 가정한다.  
이 경우에는 bio 기반 유사도로 기존 회원 중 비슷한 연결 후보를 추천한다.


In [ ]:
new_bio = "베트남 호치민 의류 OEM 업체 대표입니다. FTA로 한국 수출 진행 중."
new_bio = " ".join(okt.nouns(new_bio))

new_member_matrix = tfidf_v.transform([new_bio])
sim_scores = cosine_similarity(new_member_matrix, tfidf_matrix)
sim_indices = sim_scores.argsort()[0][::-1][:5]

print(f"추천 유저 ID: {df.iloc[sim_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[0][sim_indices]}")

df.iloc[sim_indices][["id", "member_nickname", "member_bio", "category_text"]]


In [ ]:
new_bio = "한국에서 꼼장어를 수출하는 일을 하고있는 50대 남성입니다."
new_bio = " ".join(okt.nouns(new_bio))

new_member_matrix = tfidf_v.transform([new_bio])
sim_scores = cosine_similarity(new_member_matrix, tfidf_matrix)
sim_indices = sim_scores.argsort()[0][::-1][:5]

print(f"추천 유저 ID: {df.iloc[sim_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[0][sim_indices]}")

df.iloc[sim_indices][["id", "member_nickname", "member_bio", "category_text"]]

In [ ]:
new_bio = "수출입 데이터와 글로벌 시장 흐름을 분석하며,\
기업의 해외 진출과 무역 전략을 이해하는 역량을 키워가고 있습니다.\
데이터 기반으로 시장을 읽고, 비즈니스 기회를 찾는 사람을 지향합니다."
new_bio = " ".join(okt.nouns(new_bio))

new_member_matrix = tfidf_v.transform([new_bio])
sim_scores = cosine_similarity(new_member_matrix, tfidf_matrix)
sim_indices = sim_scores.argsort()[0][::-1][:5]

print(f"추천 유저 ID: {df.iloc[sim_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[0][sim_indices]}")

df.iloc[sim_indices][["id", "member_nickname", "member_bio", "category_text"]]

In [ ]:
new_bio = "김태주 여자친구 구함"
new_bio = " ".join(okt.nouns(new_bio))

new_member_matrix = tfidf_v.transform([new_bio])
sim_scores = cosine_similarity(new_member_matrix, tfidf_matrix)
sim_indices = sim_scores.argsort()[0][::-1][:5]

print(f"추천 유저 ID: {df.iloc[sim_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[0][sim_indices]}")

df.iloc[sim_indices][["id", "member_nickname", "member_bio", "category_text"]]

### 9. bio는 없고 카테고리만 선택한 경우

자기소개는 비어 있어도, 관심 카테고리만 선택한 신규 유저는 있다.  
이 경우 카테고리 텍스트만으로도 어느 정도 추천이 가능한지 확인한다.


In [ ]:
category_names = ["수출", "물류"]
category_text = " ".join(okt.nouns(" ".join(category_names)))

new_member_matrix = tfidf_v.transform([category_text])
sim_scores = cosine_similarity(new_member_matrix, tfidf_matrix)
sim_indices = sim_scores.argsort()[0][::-1][:5]

print(f"추천 유저 ID: {df.iloc[sim_indices]['id'].values}")
print(f"상위 5개 유사도 점수: {sim_scores[0][sim_indices]}")

df.iloc[sim_indices][["id", "member_nickname", "member_bio", "category_text"]]


### 10. bio·카테고리 둘 다 없는 완전 cold start

추천 신호가 전혀 없는 경우에는 인기도가 높은 계정을 먼저 보여주는 방식으로 시작한다.  
이 단계는 추천 품질보다는 빈 화면을 피하기 위한 운영 fallback이다.


In [ ]:
df.sort_values("score", ascending=False).head(5)


### 11. 검증 — bio가 충분한 expert 유저 기준 추천 결과 확인

설명이 충분한 expert 계정을 몇 명 골라  
추천 결과가 실제로 비슷한 무역 역할과 관심사를 가진 사람으로 나오는지 눈으로 검증한다.


In [ ]:
expert_sample = df[df["member_bio"].fillna("").str.len() > 50].head(2)

for member_id in expert_sample["id"]:
    self_idx = np.where(df["id"].values == member_id)[0][0]
    sim_scores = cosine_full[self_idx]
    sim_indices = sim_scores.argsort()[::-1]

    followed_ids = follow_df[follow_df.follower_id == member_id].following_id.values
    exclude_ids = np.append(followed_ids, member_id)
    mask = ~np.isin(df.iloc[sim_indices]["id"].values, exclude_ids)
    top5_indices = sim_indices[mask][:5]

    print(f"\n=== 기준 유저 {member_id} ({df.iloc[self_idx]['member_nickname']}) ===")
    print(f"본인 bio: {df.iloc[self_idx]['member_bio']}")
    print(
        df.iloc[top5_indices][
            ["id", "member_nickname", "member_bio", "category_text", "score"]
        ].to_string(index=False)
    )


### 12. 프로젝트 적용 포인트

이 추천 로직은 GlobalGates 안에서 아래 위치에 연결할 수 있다.

1. 메인 화면 사이드바 `팔로우 추천`
2. `/friends`의 `추천 커넥션`

현재 프로젝트는 최신 가입순/미팔로우 중심 추천에 가깝다.  
이 노트북은 그 자리를 `관심사 기반 추천`으로 바꾸기 위한 1차 검증 결과물이다.


### 저장

나중에 서비스 연결 실험을 위해 벡터라이저, 유사도 행렬, 회원 id와 fallback용 점수를 저장한다.


In [ ]:
import joblib

joblib.dump(tfidf_v, 'follower_vectorizer.pkl')
joblib.dump(tfidf_matrix, 'follower_matrix.pkl')
joblib.dump(cosine_full, 'follower_cosine_full.pkl')
joblib.dump(df['id'].values, 'follower_user_ids.pkl')
joblib.dump(df['score'].values, 'follower_user_scores.pkl')
joblib.dump(category_map_df, 'follower_category_map.pkl')


## 정리

이번 추천 커넥션 로직은 GlobalGates 내부 데이터인
- 회원 bio
- 관심 카테고리
- 팔로우 관계

를 활용해 거래로 이어질 가능성이 높은 연결 후보를 추천하는 1차 실험이다.

단순 SNS 팔로우 추천이 아니라,  
무역 관심사와 역할이 비슷한 사람을 먼저 연결해  
플랫폼 안에서 신뢰 형성과 거래 가능성을 높이는 기능으로 볼 수 있다.
